In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Plot styling
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print(" All libraries imported successfully!")

 All libraries imported successfully!


In [33]:
# Load raw dataset
df = pd.read_excel("../data/raw/German Credit Data.xlsx")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (1000, 21)

Columns: ['checking account status', 'Duration in month', 'Credit history', 'Purpose', 'Credit amount', 'Savings account/bonds', 'employment', ' Installment', 'status n sex', ' Other debtors / guarantors', 'residence', 'Property', 'Age in years', 'Other installment plans', 'Housing', 'existing credits no.', 'Job', 'liability responsibles', 'Telephone', 'foreign worker', 'Category']


,checking account status,Duration in month,Credit history,Purpose,Credit amount,Savings account/bonds,employment,Installment,status n sex,Other debtors / guarantors,...,Property,Age in years,Other installment plans,Housing,existing credits no.,Job,liability responsibles,Telephone,foreign worker,Category
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [9]:
# Quick Insights\EDA using pandas profiiling
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="German Credit Data - EDA Report", explorative = True)

profile.to_file("../data/processed/eda_report.html")
print("Profiling report saved! Open data/processed/eda_report.html in your browser.")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 21/21 [00:00<00:00, 33.94it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profiling report saved! Open data/processed/eda_report.html in your browser.


In [3]:
# See all column names clearly
for i, col in enumerate(df.columns):
    print(f"{i+1}. {col}")

1. checking account status
2. Duration in month
3. Credit history
4. Purpose
5. Credit amount
6. Savings account/bonds
7. employment
8.  Installment
9. status n sex
10.  Other debtors / guarantors
11. residence
12. Property
13. Age in years
14. Other installment plans
15. Housing
16. existing credits no.
17. Job
18. liability responsibles
19. Telephone
20. foreign worker
21. Category


In [4]:
# Check data types and missing values
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nShape:", df.shape)

Data Types:
checking account status        object
Duration in month               int64
Credit history                 object
Purpose                        object
Credit amount                   int64
Savings account/bonds          object
employment                     object
 Installment                    int64
status n sex                   object
 Other debtors / guarantors    object
residence                       int64
Property                       object
Age in years                    int64
Other installment plans        object
Housing                        object
existing credits no.            int64
Job                            object
liability responsibles          int64
Telephone                      object
foreign worker                 object
Category                        int64
dtype: object

Missing Values:
checking account status        0
Duration in month              0
Credit history                 0
Purpose                        0
Credit amount              

In [5]:
# Check the target variable distribution
print("Target Variable - Category:")
print(df["Category"].value_counts())
print("\nTarget Variable Percentage:")
print(df["Category"].value_counts(normalize=True) * 100)


Target Variable - Category:
Category
1    700
2    300
Name: count, dtype: int64

Target Variable Percentage:
Category
1    70.0
2    30.0
Name: proportion, dtype: float64


In [6]:

# Check unique values in categorical columns
print("\n--- Unique Values in Categorical Columns ---")
categorical_cols = df.select_dtypes(include="object").columns
for col in categorical_cols:
    print(f"\n{col}: {df[col].unique()}")


--- Unique Values in Categorical Columns ---

checking account status: ['A11' 'A12' 'A14' 'A13']

Credit history: ['A34' 'A32' 'A33' 'A30' 'A31']

Purpose: ['A43' 'A46' 'A42' 'A40' 'A41' 'A49' 'A44' 'A45' 'A410' 'A48']

Savings account/bonds: ['A65' 'A61' 'A63' 'A64' 'A62']

employment: ['A75' 'A73' 'A74' 'A71' 'A72']

status n sex: ['A93' 'A92' 'A91' 'A94']

 Other debtors / guarantors: ['A101' 'A103' 'A102']

Property: ['A121' 'A122' 'A124' 'A123']

Other installment plans: ['A143' 'A141' 'A142']

Housing: ['A152' 'A153' 'A151']

Job: ['A173' 'A172' 'A174' 'A171']

Telephone: ['A192' 'A191']

foreign worker: ['A201' 'A202']


### ─── Decode all categorical columns using UCI data dictionary ─────────────────

In [9]:
# Checking account status
checking_map = {
    'A11': 'less_than_0_DM',
    'A12': '0_to_200_DM',
    'A13': 'greater_than_200_DM',
    'A14': 'no_checking_account'
}

checking_map

{'A11': 'less_than_0_DM',
 'A12': '0_to_200_DM',
 'A13': 'greater_than_200_DM',
 'A14': 'no_checking_account'}

In [11]:
# Credit history
credit_history_map = {
    'A30': 'no_credits_taken',
    'A31': 'all_credits_paid_duly',
    'A32': 'existing_credits_paid_duly',
    'A33': 'delay_in_paying',
    'A34': 'critical_account'
}

credit_history_map

{'A30': 'no_credits_taken',
 'A31': 'all_credits_paid_duly',
 'A32': 'existing_credits_paid_duly',
 'A33': 'delay_in_paying',
 'A34': 'critical_account'}

In [13]:

# Purpose
purpose_map = {
    'A40': 'car_new',
    'A41': 'car_used',
    'A42': 'furniture',
    'A43': 'radio_tv',
    'A44': 'domestic_appliances',
    'A45': 'repairs',
    'A46': 'education',
    'A47': 'vacation',
    'A48': 'retraining',
    'A49': 'business',
    'A410': 'others'
}

purpose_map

{'A40': 'car_new',
 'A41': 'car_used',
 'A42': 'furniture',
 'A43': 'radio_tv',
 'A44': 'domestic_appliances',
 'A45': 'repairs',
 'A46': 'education',
 'A47': 'vacation',
 'A48': 'retraining',
 'A49': 'business',
 'A410': 'others'}

In [14]:

# Savings account/bonds
savings_map = {
    'A61': 'less_than_100_DM',
    'A62': '100_to_500_DM',
    'A63': '500_to_1000_DM',
    'A64': 'greater_than_1000_DM',
    'A65': 'unknown_no_savings'
}

savings_map

{'A61': 'less_than_100_DM',
 'A62': '100_to_500_DM',
 'A63': '500_to_1000_DM',
 'A64': 'greater_than_1000_DM',
 'A65': 'unknown_no_savings'}

In [28]:

# Employment
employment_map = {
    'A71': 'unemployed',
    'A72': 'less_than_1_year',
    'A73': '1_to_4_years',
    'A74': '4_to_7_years',
    'A75': 'greater_than_7_years'
}

employment_map

{'A71': 'unemployed',
 'A72': 'less_than_1_year',
 'A73': '1_to_4_years',
 'A74': '4_to_7_years',
 'A75': 'greater_than_7_years'}

In [16]:
# Status and sex
status_sex_map = {
    'A91': 'male_divorced_separated',
    'A92': 'female_divorced_separated_married',
    'A93': 'male_single',
    'A94': 'male_married_widowed',
    'A95': 'female_single'
}
status_sex_map

{'A91': 'male_divorced_separated',
 'A92': 'female_divorced_separated_married',
 'A93': 'male_single',
 'A94': 'male_married_widowed',
 'A95': 'female_single'}

In [17]:

# Other debtors / guarantors
other_debtors_map = {
    'A101': 'none',
    'A102': 'co_applicant',
    'A103': 'guarantor'
}

other_debtors_map

{'A101': 'none', 'A102': 'co_applicant', 'A103': 'guarantor'}

In [19]:
# Property
property_map = {
    'A121': 'real_estate',
    'A122': 'savings_insurance',
    'A123': 'car_other',
    'A124': 'unknown_no_property'
}

property_map

{'A121': 'real_estate',
 'A122': 'savings_insurance',
 'A123': 'car_other',
 'A124': 'unknown_no_property'}

In [20]:
# Other installment plans
installment_plans_map = {
    'A141': 'bank',
    'A142': 'stores',
    'A143': 'none'
}

installment_plans_map

{'A141': 'bank', 'A142': 'stores', 'A143': 'none'}

In [21]:
# Housing
housing_map = {
    'A151': 'rent',
    'A152': 'own',
    'A153': 'for_free'
}

housing_map

{'A151': 'rent', 'A152': 'own', 'A153': 'for_free'}

In [22]:
# Job
job_map = {
    'A171': 'unemployed_unskilled_non_resident',
    'A172': 'unskilled_resident',
    'A173': 'skilled_employee',
    'A174': 'management_self_employed'
}

job_map

{'A171': 'unemployed_unskilled_non_resident',
 'A172': 'unskilled_resident',
 'A173': 'skilled_employee',
 'A174': 'management_self_employed'}

In [23]:
# Telephone
telephone_map = {
    'A191': 'none',
    'A192': 'yes_registered'
}

telephone_map

{'A191': 'none', 'A192': 'yes_registered'}

In [24]:
# Foreign worker
foreign_worker_map = {
    'A201': 'yes',
    'A202': 'no'
}

foreign_worker_map

{'A201': 'yes', 'A202': 'no'}

In [29]:
# ─── Apply all mappings to the dataframe ──────────────────────────────────────
df['checking account status'] = df['checking account status'].map(checking_map)
df['Credit history'] = df['Credit history'].map(credit_history_map)
df['Purpose'] = df['Purpose'].map(purpose_map)
df['Savings account/bonds'] = df['Savings account/bonds'].map(savings_map)
df['employment'] = df['employment'].map(employment_map)
df['status n sex'] = df['status n sex'].map(status_sex_map)
df[' Other debtors / guarantors'] = df[' Other debtors / guarantors'].map(other_debtors_map)
df['Property'] = df['Property'].map(property_map)
df['Other installment plans'] = df['Other installment plans'].map(installment_plans_map)
df['Housing'] = df['Housing'].map(housing_map)
df['Job'] = df['Job'].map(job_map)
df['Telephone'] = df['Telephone'].map(telephone_map)
df['foreign worker'] = df['foreign worker'].map(foreign_worker_map)

In [31]:
df.tail(10)

,checking account status,Duration in month,Credit history,Purpose,Credit amount,Savings account/bonds,employment,Installment,status n sex,Other debtors / guarantors,...,Property,Age in years,Other installment plans,Housing,existing credits no.,Job,liability responsibles,Telephone,foreign worker,Category
990,NaN,12,NaN,NaN,3565,NaN,NaN,2,NaN,NaN,...,NaN,37,NaN,NaN,2,NaN,2,NaN,NaN,1
991,NaN,15,NaN,NaN,1569,NaN,NaN,4,NaN,NaN,...,NaN,34,NaN,NaN,1,NaN,2,NaN,NaN,1
992,NaN,18,NaN,NaN,1936,NaN,NaN,2,NaN,NaN,...,NaN,23,NaN,NaN,2,NaN,1,NaN,NaN,1
993,NaN,36,NaN,NaN,3959,NaN,NaN,4,NaN,NaN,...,NaN,30,NaN,NaN,1,NaN,1,NaN,NaN,1
994,NaN,12,NaN,NaN,2390,NaN,NaN,4,NaN,NaN,...,NaN,50,NaN,NaN,1,NaN,1,NaN,NaN,1
995,NaN,12,NaN,NaN,1736,NaN,NaN,3,NaN,NaN,...,NaN,31,NaN,NaN,1,NaN,1,NaN,NaN,1
996,NaN,30,NaN,NaN,3857,NaN,NaN,4,NaN,NaN,...,NaN,40,NaN,NaN,1,NaN,1,NaN,NaN,1
997,NaN,12,NaN,NaN,804,NaN,NaN,4,NaN,NaN,...,NaN,38,NaN,NaN,1,NaN,1,NaN,NaN,1
998,NaN,45,NaN,NaN,1845,NaN,NaN,4,NaN,NaN,...,NaN,23,NaN,NaN,1,NaN,1,NaN,NaN,2
999,NaN,45,NaN,NaN,4576,NaN,NaN,3,NaN,NaN,...,NaN,27,NaN,NaN,1,NaN,1,NaN,NaN,1


In [32]:


# ─── Recode target variable: 1=good becomes 0, 2=bad becomes 1 ───────────────
df['Category'] = df['Category'].map({1: 0, 2: 1})

print("All categorical columns decoded successfully!")
print("\nTarget variable recoded: 0 = Good Credit Risk, 1 = Bad Credit Risk")
print("\nNew target distribution:")
print(df['Category'].value_counts())
print("\nFirst 3 rows after decoding:")
df.head(3)

All categorical columns decoded successfully!

Target variable recoded: 0 = Good Credit Risk, 1 = Bad Credit Risk

New target distribution:
Category
0    700
1    300
Name: count, dtype: int64

First 3 rows after decoding:


,checking account status,Duration in month,Credit history,Purpose,Credit amount,Savings account/bonds,employment,Installment,status n sex,Other debtors / guarantors,...,Property,Age in years,Other installment plans,Housing,existing credits no.,Job,liability responsibles,Telephone,foreign worker,Category
0,NaN,6,NaN,NaN,1169,NaN,NaN,4,NaN,NaN,...,NaN,67,NaN,NaN,2,NaN,1,NaN,NaN,0
1,NaN,48,NaN,NaN,5951,NaN,NaN,2,NaN,NaN,...,NaN,22,NaN,NaN,1,NaN,1,NaN,NaN,1
2,NaN,12,NaN,NaN,2096,NaN,NaN,2,NaN,NaN,...,NaN,49,NaN,NaN,1,NaN,2,NaN,NaN,0
